# [Optional] Adversarial ML

This optional FGSM attack notebook shows working with pretrained models and better quality images. We also explore a 'targeted' attack, which means trying to get an image of class X misclassified specifically as an image of class Y.

## FGSM Attack (a reminder)

The *Fast Gradient Sign Attack* is one of the key adversarial attacks to cause untargeted or targeted misclassifications. 

Rather than minimizing the loss by adjusting the weights based on backpropagated gradients, the FGSM seeks to adjust the input data to *maximize* the loss based on the same backpropagated gradients. There is a simple PyTorch tutorial [here](https://docs.pytorch.org/tutorials/beginner/fgsm_tutorial.html) to run through the algorithm.






In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet50, ResNet50_Weights
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import requests
from io import BytesIO
import json

We will load a pretrained ResNet50, a popular model for computer vision tasks. Our images come from the Imagenet dataset and some downloaded images from which we will make adversarial examples.

In [ ]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pretrained ResNet50 model
model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
model.eval()
model = model.to(device)

# ImageNet normalization
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])

# Transform for preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize
])

# Transform for display (denormalize)
denormalize = transforms.Normalize(mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
                                 std=[1/0.229, 1/0.224, 1/0.225])

# Load ImageNet class labels
IMAGENET_CLASSES_URL = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
try:
    response = requests.get(IMAGENET_CLASSES_URL)
    imagenet_classes = response.text.strip().split('\n')
except:
    # Fallback - basic classes for demonstration
    imagenet_classes = [f"class_{i}" for i in range(1000)]


In [ ]:
def load_sample_images():
    """Load sample images from reliable URLs - real images only"""

    # Set proper headers to avoid 403 errors
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
    }

    # Reliable image URLs - using direct links that don't require special headers
    image_urls = [
        # "https://picsum.photos/400/400?random=1",
        # "https://picsum.photos/400/400?random=2",
        # "https://picsum.photos/400/400?random=3",
        # "https://picsum.photos/400/400?random=4",
        # "https://picsum.photos/400/400?random=5",
        # "https://picsum.photos/400/400?random=6",
        "https://picsum.photos/id/219/400/400",
        "https://picsum.photos/id/237/400/400",
        "https://picsum.photos/id/32/400/400",
        "https://picsum.photos/id/40/400/400",
        "https://picsum.photos/id/58/400/400",
        "https://picsum.photos/id/230/400/400",
    ]

    images = []
    print("Attempting to load real images from web...")

    for i, url in enumerate(image_urls):
        if len(images) >= 6:  # We only need 6 images
            break

        try:
            print(f"Trying image {len(images)+1}: {url}")
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()

            img = Image.open(BytesIO(response.content)).convert('RGB')

            # Verify it's a real image (has reasonable dimensions)
            if img.size[0] >= 100 and img.size[1] >= 100:
                images.append(img)
                print(f"✓ Successfully loaded real image {len(images)}")
            else:
                print(f"✗ Image too small: {img.size}")

        except Exception as e:
            print(f"✗ Failed to load from {url}: {e}")
            continue

    if len(images) < 6:
        print(f"\nWARNING: Only loaded {len(images)} images out of 6 requested.")
        print("The demo will work with fewer images, but results may be limited.")

    print(f"\nSuccessfully prepared {len(images)} images for adversarial attack demo!")
    return images

In [ ]:
def fgsm_attack(model, image, epsilon, target_class=None):
    """
    Perform FGSM attack on a single image - proper implementation for imperceptible perturbations

    Args:
        model: The neural network model
        image: Input image tensor (1, C, H, W) - should be normalized
        epsilon: Attack strength in normalized space (much smaller values needed)
        target_class: If specified, perform targeted attack

    Returns:
        adversarial_image: Adversarial example
        original_pred: Original prediction
        adv_pred: Adversarial prediction
    """
    # Ensure image requires gradient
    image = image.clone().detach().requires_grad_(True)

    # Forward pass
    output = model(image)
    original_pred = output.argmax(dim=1).item()
    print(f"Original prediction for image: {original_pred}")

    # Calculate loss
    if target_class is not None:
        # Targeted attack - minimize loss for target class
        loss = F.cross_entropy(output, torch.tensor([target_class]).to(device))
        loss = -loss  # Minimize loss = maximize negative loss
    else:
        # Untargeted attack - maximize loss for true class
        loss = F.cross_entropy(output, torch.tensor([original_pred]).to(device))

    # Backward pass
    model.zero_grad()
    loss.backward()

    # Generate adversarial example
    data_grad = image.grad.data
    sign_data_grad = data_grad.sign()
    adversarial_image = image + epsilon * sign_data_grad

    # CRITICAL: Clip in normalized space to maintain valid range
    # For ImageNet normalization, valid range is roughly [-2.1, 2.6]
    # But we need to be more careful about the actual bounds
    mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1).to(device)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1).to(device)

    # Convert to [0,1] range, clip, then convert back
    adversarial_image_01 = adversarial_image * std + mean
    adversarial_image_01 = torch.clamp(adversarial_image_01, 0, 1)
    adversarial_image = (adversarial_image_01 - mean) / std

    # Get adversarial prediction
    with torch.no_grad():
        adv_output = model(adversarial_image)
        adv_pred = adv_output.argmax(dim=1).item()

    return adversarial_image, original_pred, adv_pred


In [ ]:
def preprocess_image(pil_image):
    """Convert PIL image to tensor and preprocess."""
    # Apply preprocessing
    tensor = preprocess(pil_image).unsqueeze(0).to(device)
    return tensor

def tensor_to_pil(tensor):
    """Convert tensor back to PIL image for display"""
    # Denormalize
    tensor = denormalize(tensor.squeeze(0).cpu())
    # Clamp to [0, 1] - ensure pixels stay in valid range
    tensor = torch.clamp(tensor, 0, 1)
    # Convert to PIL displayable image
    to_pil = transforms.ToPILImage()
    return to_pil(tensor)



### Adversarial grid

Here is the main attack. For each image, we take the original image, test different attack strenghts by tweaking the epsilon value, generating adversarial examples and measuring their success in fooling the classifier.

In [ ]:
def create_adversarial_grid(images, epsilon_values=[0.007, 0.01, 0.02, 0.05, 0.1]):
    """Create a grid showing original and adversarial images"""
    n_images = len(images)
    n_epsilons = len(epsilon_values)

    fig, axes = plt.subplots(n_images, n_epsilons + 1, figsize=(20, 4 * n_images))
    fig.suptitle('FGSM Adversarial Examples', fontsize=16)

    results = []

    for i, pil_img in enumerate(images):
        # Preprocess image
        img_tensor = preprocess_image(pil_img)

        # Original prediction
        with torch.no_grad():
            orig_output = model(img_tensor)
            orig_pred = orig_output.argmax(dim=1).item()
            orig_conf = F.softmax(orig_output, dim=1).max().item()

        # Display original image
        ax = axes[i, 0] if n_images > 1 else axes[0]
        ax.imshow(pil_img)
        ax.set_title(f'Original\n{imagenet_classes[orig_pred][:20]}\nConf: {orig_conf:.3f}')
        ax.axis('off')

        row_results = {'original': {'pred': orig_pred, 'conf': orig_conf}}

        # Generate adversarial examples for different epsilon values
        for j, epsilon in enumerate(epsilon_values):
            adv_tensor, orig_pred_check, adv_pred = fgsm_attack(model, img_tensor, epsilon)

            # Get adversarial confidence
            with torch.no_grad():
                adv_output = model(adv_tensor)
                adv_conf = F.softmax(adv_output, dim=1).max().item()

            # Convert back to PIL for display
            adv_pil = tensor_to_pil(adv_tensor)

            # Display adversarial image
            ax = axes[i, j + 1] if n_images > 1 else axes[j + 1]
            ax.imshow(adv_pil)

            # Color code title based on success
            color = 'red' if adv_pred != orig_pred else 'blue'
            ax.set_title(f'ε={epsilon}\n{imagenet_classes[adv_pred][:20]}\nConf: {adv_conf:.3f}',
                        color=color)
            ax.axis('off')

            row_results[f'eps_{epsilon}'] = {
                'pred': adv_pred,
                'conf': adv_conf,
                'success': adv_pred != orig_pred
            }

        results.append(row_results)

    plt.tight_layout()
    plt.show()

    return results


In [ ]:
def analyze_attack_success(results):
    """Analyze and display attack success rates"""
    epsilon_values = [0.007, 0.01, 0.02]
    success_rates = []

    for epsilon in epsilon_values:
        successes = sum(1 for r in results if r[f'eps_{epsilon}']['success'])
        success_rate = successes / len(results)
        success_rates.append(success_rate)

    # Plot success rates
    plt.figure(figsize=(10, 6))
    plt.bar(range(len(epsilon_values)), success_rates)
    plt.xlabel('Epsilon Value')
    plt.ylabel('Attack Success Rate')
    plt.title('FGSM Attack Success Rate vs Epsilon')
    plt.xticks(range(len(epsilon_values)), epsilon_values)
    plt.ylim(0, 1)

    # Add value labels on bars
    for i, v in enumerate(success_rates):
        plt.text(i, v + 0.01, f'{v:.2f}', ha='center', va='bottom')

    plt.tight_layout()
    plt.show()

    print("Attack Success Summary:")
    for i, epsilon in enumerate(epsilon_values):
        print(f"ε = {epsilon}: {success_rates[i]:.2f} success rate")


In [ ]:
def create_demo():
    """Show image misclassification with imperceptible perturbations"""
    # Load a sample image
    sample_images = load_sample_images()
    pil_img = sample_images[0]  # Use first image

    # Very small epsilon values like the original paper
    epsilons = [0.007, 0.014]  # These are the actual values from Goodfellow et al.

    fig, axes = plt.subplots(len(epsilons), 4, figsize=(16, 8))
    fig.suptitle('Imperceptible Adversarial Perturbations', fontsize=14)

    for i, epsilon in enumerate(epsilons):
        img_tensor = preprocess_image(pil_img)

        # Original prediction
        with torch.no_grad():
            orig_output = model(img_tensor)
            orig_pred = orig_output.argmax(dim=1).item()
            orig_conf = F.softmax(orig_output, dim=1)[0, orig_pred].item()

        # Generate adversarial example
        adv_tensor, _, adv_pred = fgsm_attack(model, img_tensor, epsilon)

        # Get adversarial confidence
        with torch.no_grad():
            adv_output = model(adv_tensor)
            adv_conf = F.softmax(adv_output, dim=1)[0, adv_pred].item()

        # Original image (resize for consistent display)
        orig_resized = pil_img.resize((224, 224))
        axes[i, 0].imshow(orig_resized)
        axes[i, 0].set_title(f'Original\n{imagenet_classes[orig_pred][:15]}\n{orig_conf:.3f} confidence')
        axes[i, 0].axis('off')

        # Adversarial image
        adv_pil = tensor_to_pil(adv_tensor)
        axes[i, 1].imshow(adv_pil)
        success_color = 'red' if adv_pred != orig_pred else 'blue'
        axes[i, 1].set_title(f'Adversarial (ε={epsilon})\n{imagenet_classes[adv_pred][:15]}\n{adv_conf:.3f} confidence',
                           color=success_color)
        axes[i, 1].axis('off')

        # Perturbation (highly amplified)
        perturbation = adv_tensor - img_tensor
        # Convert perturbation to visible range
        pert_vis = perturbation.squeeze(0).cpu().detach()
        # Amplify by 50x and normalize to [0,1] for visibility
        pert_vis = pert_vis * 50
        pert_vis = (pert_vis - pert_vis.min()) / (pert_vis.max() - pert_vis.min())
        axes[i, 2].imshow(pert_vis.permute(1, 2, 0))
        axes[i, 2].set_title(f'Perturbation\n(50x amplified)')
        axes[i, 2].axis('off')

        # Difference in pixel space (resize original to match processed size)
        orig_resized = pil_img.resize((224, 224))
        orig_pixels = np.array(orig_resized).astype(float)
        adv_pixels = np.array(adv_pil).astype(float)
        diff = np.abs(adv_pixels - orig_pixels)
        # Amplify difference for visibility
        diff_vis = np.clip(diff * 10, 0, 255)
        axes[i, 3].imshow(diff_vis.astype(np.uint8))
        axes[i, 3].set_title(f'Pixel Difference\n(10x amplified)\nMax diff: {diff.max():.1f}')
        axes[i, 3].axis('off')

    plt.tight_layout()
    plt.show()

    print(f"\nKey insight: With ε={epsilons[0]}, max pixel change is ~{diff.max():.1f}/255")
    print("This is barely perceptible to humans but can fool the network!")

# Add this to the main execution section
if __name__ == "__main__":
    print("Loading sample images...")
    sample_images = load_sample_images()

    print("Generating adversarial examples with imperceptible perturbations...")
    results = create_adversarial_grid(sample_images)

    print("Analyzing attack success...")
    analyze_attack_success(results)

    print("\nCreating demonstration...")
    create_demo()

    print("\nDemonstration complete!")

# Main execution
if __name__ == "__main__":
    print("Loading sample images...")
    sample_images = load_sample_images()

    print("Generating adversarial examples...")
    results = create_adversarial_grid(sample_images)

    print("Analyzing attack success...")
    analyze_attack_success(results)


In [ ]:
def targeted_attack_demo(image_index=0, target_class=281):  # tabby cat
    """Demonstrate targeted FGSM attack"""
    if image_index >= len(sample_images):
        print("Invalid image index")
        return

    pil_img = sample_images[image_index]
    img_tensor = preprocess_image(pil_img)

    # Original prediction
    with torch.no_grad():
        orig_output = model(img_tensor)
        orig_pred = orig_output.argmax(dim=1).item()

    epsilons = [0.007, 0.05, 0.1, 0.15, 0.2, 0.3]
    
    fig, axes = plt.subplots(1, len(epsilons) + 1, figsize=(16, 4))
    fig.suptitle(f'Targeted Attack: Trying to make model predict "{imagenet_classes[target_class]}"')

    # Original - resize to match processed size for consistent display
    pil_img_resized = pil_img.resize((224, 224))
    axes[0].imshow(pil_img_resized)
    axes[0].set_title(f'Original\n{imagenet_classes[orig_pred][:20]}')
    axes[0].axis('off')

    for i, epsilon in enumerate(epsilons):
        adv_tensor, _, adv_pred = fgsm_attack(model, img_tensor, epsilon, target_class)
        adv_pil = tensor_to_pil(adv_tensor)

        color = 'green' if adv_pred == target_class else 'red'
        axes[i + 1].imshow(adv_pil)
        axes[i + 1].set_title(f'ε={epsilon}\n{imagenet_classes[adv_pred][:20]}', color=color)
        axes[i + 1].axis('off')

    plt.tight_layout()
    plt.show()



In [ ]:
def perturbation_visualization(image_index=0, epsilon=0.007):
    """Visualize the perturbation itself"""
    if image_index >= len(sample_images):
        print("Invalid image index")
        return

    pil_img = sample_images[image_index]
    img_tensor = preprocess_image(pil_img)

    # Generate adversarial example
    adv_tensor, orig_pred, adv_pred = fgsm_attack(model, img_tensor, epsilon)

    # Calculate perturbation
    perturbation = adv_tensor - img_tensor

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    fig.suptitle(f'Perturbation Visualization (ε={epsilon})')

    # Original - resize to match processed size for consistent display
    pil_img_resized = pil_img.resize((224, 224))
    axes[0].imshow(pil_img_resized)
    axes[0].set_title(f'Original\n{imagenet_classes[orig_pred][:20]}')
    axes[0].axis('off')

    # Perturbation (amplified for visibility)
    pert_display = perturbation.squeeze(0).cpu().detach()
    pert_display = (pert_display - pert_display.min()) / (pert_display.max() - pert_display.min())
    axes[1].imshow(pert_display.permute(1, 2, 0))
    axes[1].set_title('Perturbation\n(Amplified)')
    axes[1].axis('off')

    # Adversarial
    adv_pil = tensor_to_pil(adv_tensor)
    axes[2].imshow(adv_pil)
    axes[2].set_title(f'Adversarial\n{imagenet_classes[adv_pred][:20]}')
    axes[2].axis('off')

    # Difference (visible perturbation) - now both images are 224x224
    diff_img = np.array(adv_pil).astype(float) - np.array(pil_img_resized).astype(float)
    diff_img = (diff_img + 255) / 2  # Normalize to [0, 255]
    axes[3].imshow(diff_img.astype(np.uint8))
    axes[3].set_title('Visible Difference\n(Amplified)')
    axes[3].axis('off')

    plt.tight_layout()
    plt.show()

# Uncomment to run additional demonstrations
targeted_attack_demo(1, 281)  # Try to make it predict "tabby cat"
perturbation_visualization(0, 0.1)  # Visualize perturbations

In [ ]:
targeted_attack_demo(1, 290)

In [ ]:
targeted_attack_demo(5, 386)